# 🦙 Llama-3.1-8B-Stheno-v3.4 on Free Google Colab + Ngrok for ZetaForge

This notebook configures a free Google Colab GPU runtime to download and run the **Llama-3.1-8B-Stheno-v3.4-Q4_K_M GGUF** with CUDA-accelerated **KoboldCpp**, expose it through the reserved ngrok hostname, and verify the exact OpenAI-compatible API that ZetaForge uses.

### One-time setup
1. In **Runtime → Change runtime type**, select a GPU (a T4 is ideal when available).
2. In the Colab left sidebar, open **Secrets** (key icon) and add a secret named `NGROK_AUTHTOKEN`. This is the recommended way to avoid pasting the token every run.
3. Run **Runtime → Run all**. Colab intentionally does not auto-execute arbitrary notebooks just because a URL was opened; this notebook therefore makes the normal Run all flow fully automatic after that one user action.

If the secret is absent, the configuration cell securely prompts for the token instead. The token is never placed in a URL or committed to GitHub.


In [ ]:
#@title 1. Check GPU Environment
import torch, sys, subprocess

if not torch.cuda.is_available():
    raise SystemError("❌ No GPU detected. Choose Runtime → Change runtime type → GPU and run again.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ CUDA GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=False)


In [ ]:
#@title 2. Install Tools & Download KoboldCpp
import os
print("📦 Installing aria2 and Python utilities…")
!apt-get update -y -qq && apt-get install -y -qq aria2
!pip install -q pyngrok requests huggingface_hub

print("⚙️ Downloading the latest KoboldCpp CUDA binary…")
!curl -fLo koboldcpp https://github.com/LostRuins/koboldcpp/releases/latest/download/koboldcpp-linux-x64
!chmod +x koboldcpp
print("✅ KoboldCpp ready")


In [ ]:
#@title 3. Download Llama-3.1-8B-Stheno-v3.4-Q4_K_M
import os

MODEL_REPO_URL = "https://huggingface.co/bartowski/Llama-3.1-8B-Stheno-v3.4-GGUF/resolve/main/Llama-3.1-8B-Stheno-v3.4-Q4_K_M.gguf"
MODEL_FILENAME = "Llama-3.1-8B-Stheno-v3.4-Q4_K_M.gguf"
MODEL_PATH = f"/content/{MODEL_FILENAME}"

if not os.path.exists(MODEL_PATH):
    print(f"📥 Downloading {MODEL_FILENAME} (~4.9 GB)…")
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{MODEL_REPO_URL}" -d /content -o "{MODEL_FILENAME}"
else:
    print(f"✅ Model already present: {MODEL_PATH}")

if not os.path.exists(MODEL_PATH) or os.path.getsize(MODEL_PATH) < 4_000_000_000:
    raise RuntimeError("Model download appears incomplete.")
print(f"✅ Model size: {os.path.getsize(MODEL_PATH)/(1024**3):.2f} GiB")


In [ ]:
#@title 4. Configure Ngrok Securely
import getpass, os, requests, subprocess, time
from pyngrok import conf, ngrok

RESERVED_DOMAIN = "paralegal-pampers-chevron.ngrok-free.dev" #@param {type:"string"}
LOCAL_PORT = 5001

NGROK_AUTHTOKEN = ""
try:
    from google.colab import userdata
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN") or ""
except Exception:
    pass

if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = os.environ.get("NGROK_AUTHTOKEN", "")
if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = getpass.getpass("🔑 Ngrok Authtoken (not echoed): ")

NGROK_AUTHTOKEN = NGROK_AUTHTOKEN.strip()
if not NGROK_AUTHTOKEN:
    raise ValueError("No Ngrok Authtoken supplied.")

# Kill stale local ngrok processes from an interrupted Colab session.
try:
    ngrok.kill()
except Exception:
    pass
subprocess.run(["pkill", "-9", "ngrok"], stderr=subprocess.DEVNULL, check=False)
time.sleep(1)

# If the reserved hostname is still held by an orphan endpoint, release it.
api_headers = {"Authorization": f"Bearer {NGROK_AUTHTOKEN}", "Ngrok-Version": "2"}
try:
    r = requests.get("https://api.ngrok.com/endpoints", headers=api_headers, timeout=10)
    if r.ok:
        for ep in r.json().get("endpoints", []):
            if RESERVED_DOMAIN in ep.get("public_url", ""):
                ep_id = ep.get("id")
                if ep_id:
                    print(f"🧹 Releasing orphan ngrok endpoint {ep_id}…")
                    requests.delete(f"https://api.ngrok.com/endpoints/{ep_id}", headers=api_headers, timeout=10)
except Exception as e:
    print(f"ℹ️ Remote endpoint cleanup skipped: {e}")

conf.get_default().auth_token = NGROK_AUTHTOKEN
print(f"✅ Ngrok credentials loaded. Reserved domain: {RESERVED_DOMAIN}")


In [ ]:
#@title 5. Start KoboldCpp + Ngrok and Verify the ZetaForge API
import os, subprocess, time, requests, json
from pathlib import Path
from pyngrok import ngrok

CONTEXT_SIZE = 8192 #@param [4096, 8192, 16384] {type:"raw"}
GPU_LAYERS = 33
API_BASE = f"https://{RESERVED_DOMAIN}/v1"
LOCAL_BASE = f"http://127.0.0.1:{LOCAL_PORT}/v1"

# Start KoboldCpp in the background so the remaining verification code can run
# in this same notebook execution. This fixes the old foreground-cell design,
# where the verification cell could not run until the server was stopped.
server_cmd = [
    "./koboldcpp", "--model", MODEL_PATH,
    "--port", str(LOCAL_PORT),
    "--gpulayers", str(GPU_LAYERS),
    "--contextsize", str(CONTEXT_SIZE),
    "--flashattention", "--smartcontext"
]
print("🔥 Starting KoboldCpp in background…")
server_log = open("/content/koboldcpp.log", "a", buffering=1)
server_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)

# Wait for the local OpenAI endpoint before opening the public tunnel.
ready = False
for _ in range(90):
    if server_proc.poll() is not None:
        tail = Path("/content/koboldcpp.log").read_text(errors="replace")[-4000:]
        raise RuntimeError("KoboldCpp exited during startup.\n" + tail)
    try:
        rr = requests.get(f"{LOCAL_BASE}/models", timeout=2)
        if rr.ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)
if not ready:
    raise TimeoutError("KoboldCpp did not expose /v1/models within 90 seconds. Check /content/koboldcpp.log.")
print("✅ Local OpenAI-compatible endpoint is ready")

# Create the reserved ngrok tunnel only after the model server is alive.
try:
    ngrok.kill()
except Exception:
    pass
try:
    tunnel = ngrok.connect(LOCAL_PORT, domain=RESERVED_DOMAIN)
except Exception:
    tunnel = ngrok.connect(LOCAL_PORT, domain=RESERVED_DOMAIN, pooling_enabled=True)

PUBLIC_BASE = tunnel.public_url.rstrip("/") + "/v1"
print("🚀 NGROK TUNNEL ACTIVE")
print("🌐 Public API:", PUBLIC_BASE)

headers = {"Content-Type":"application/json", "ngrok-skip-browser-warning":"true"}

# Stage 1: tunnel/model discovery.
models_res = requests.get(PUBLIC_BASE + "/models", headers=headers, timeout=15)
models_res.raise_for_status()
models_data = models_res.json()
model_id = (models_data.get("data") or [{}])[0].get("id") or "Llama-3.1-8B-Stheno-v3.4"
print("✅ /v1/models reachable")
print("🧠 Advertised model:", model_id)

# Stage 2: real inference request — this is the same route ZetaForge calls.
payload = {
    "model": model_id,
    "messages": [{"role":"user", "content":"Reply with exactly: ZetaForge connection OK"}],
    "temperature": 0,
    "max_tokens": 12,
    "stream": False
}
chat_res = requests.post(PUBLIC_BASE + "/chat/completions", headers=headers, json=payload, timeout=90)
chat_res.raise_for_status()
chat_data = chat_res.json()
reply = ((chat_data.get("choices") or [{}])[0].get("message") or {}).get("content", "").strip()
print("✅ /v1/chat/completions reachable")
print("🤖 Test response:", reply)
print("=" * 72)
print("ZetaForge endpoint:", PUBLIC_BASE)
print("Keep this notebook's final cell running while using ZetaForge.")
print("=" * 72)


In [ ]:
#@title 6. Keep the Colab Server Alive
# This final cell intentionally stays running. Stop this cell/runtime to shut down
# the server and ngrok tunnel.
import time, requests, os

print("🟢 ZetaForge Colab server is live. Press Stop/Interrupt to shut it down.")
while True:
    try:
        r = requests.get(f"http://127.0.0.1:{LOCAL_PORT}/v1/models", timeout=5)
        if not r.ok:
            print(f"⚠️ Local API returned HTTP {r.status_code}; inspect /content/koboldcpp.log")
    except Exception as e:
        print("⚠️ Local API check failed:", e)
    time.sleep(60)
